# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnsoundMouse/flyrankaiw01_research_question/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Task shape:** predicting `underperforms` (my Week 1-2 proxy: is this page's CTR below its own
position tier's median?) is a **yes/no question with an observed-from-current-data label** — so
per the skill's table, I start with **Logistic Regression** (readable), then check whether
**Random Forest** (stronger, nonlinear) actually earns its extra complexity.

Crucially, this is *not* the same task as "recreate the Week 4 rule." The Week 4 rule computes
`ctr_gap` directly from `ctr` — if I gave a model `ctr` as a feature, it would trivially
reconstruct the label with ~100% accuracy, which teaches nothing. So the real, useful task here
is: **can other signals about a page (its content, freshness, engagement, position — never its
own CTR) predict whether it's likely underperforming its tier, without seeing the CTR that
defines the label?** That's a genuinely open question — I checked first, and `position_tier`
alone is close to a coin flip within the eligible pool (48-50% underperform rate in every tier),
so position alone clearly isn't the answer, and there's real room for other features to help.

**Fair baseline for this comparison:** ranking by `avg_position` alone (best position first) —
this reduces the Week 4 rule to the one signal it can offer *without* touching CTR, which is
exactly the version I can honestly compare a model against on the same data and metric.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/UnsoundMouse/flyrankaiw01_research_question/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

eligible = df[(df["impressions_90d"] > 0) & (df["avg_position"] > 0)].copy()
visible = eligible[eligible["impressions_90d"] >= 500]
tier_median = visible.groupby("position_tier")["ctr"].median()
eligible = eligible.merge(tier_median.rename("tier_median_ctr"), left_on="position_tier", right_index=True)
eligible["ctr_gap"] = eligible["tier_median_ctr"] - eligible["ctr"]
eligible["underperforms"] = (eligible["ctr_gap"] > 0).astype(int)

# same pool as the Week 4 baseline
pool = eligible[(eligible["impressions_90d"] >= 500) & (eligible["avg_position"] <= 20)].copy().reset_index(drop=True)

print(f"Pool: {len(pool):,} rows | base rate (underperforms=1): {pool['underperforms'].mean():.3f}")
print("Per-tier underperform rate (checking position_tier isn't already the answer):")
print(pool.groupby("position_tier")["underperforms"].mean().round(3))

Pool: 12,023 rows | base rate (underperforms=1): 0.489
Per-tier underperform rate (checking position_tier isn't already the answer):
position_tier
page_1      0.496
page_3_5    0.312
striking    0.480
top_3       0.480
Name: underperforms, dtype: float64


## 2. Split design

**GroupShuffleSplit by `client_id`, 75/25.** A client's pages must never span train and test —
otherwise the model could learn client-specific quirks (a particular CMS, a particular content
team's style) rather than generalizable signal, and the test score would be optimistic. Fixed
`random_state=42` throughout, so rerunning reproduces the same numbers.

**Excluded from features, always:** `ctr`, `clicks_90d`, `tier_median_ctr`, `ctr_gap` (the label
itself and everything it's built from), plus `trend_pct`/`trend_direction` (label-trap columns
per the data dictionary, never features regardless of task).

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

num_feats = ["avg_position", "word_count", "content_age_days", "days_since_last_update",
             "engagement_rate", "scroll_rate", "ai_traffic_pct"]
cat_feats = ["content_type", "main_intent", "freshness_tier"]

# sessions_90d included here so Pass 1 (Cell 6) can use it -- it gets dropped in Pass 2 by column selection, not by absence
all_num_feats = num_feats + ["sessions_90d"]

y = pool["underperforms"].to_numpy()
groups = pool["client_id"].to_numpy()
X = pool[all_num_feats + cat_feats].copy()
for c in all_num_feats:
    X[c] = X[c].fillna(X[c].median())

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
ytr, yte = y[train_idx], y[test_idx]
pool_te = pool.iloc[test_idx].reset_index(drop=True)

train_clients = set(pool.iloc[train_idx]["client_id"])
test_clients = set(pool.iloc[test_idx]["client_id"])
print(f"Train: {len(Xtr):,} rows, {len(train_clients)} clients")
print(f"Test:  {len(Xte):,} rows, {len(test_clients)} clients")
print(f"Client overlap between train/test: {len(train_clients & test_clients)} (must be 0)")
print(f"Test base rate: {yte.mean():.3f}")

Train: 11,199 rows, 21 clients
Test:  824 rows, 7 clients
Client overlap between train/test: 0 (must be 0)
Test base rate: 0.439


## 3. Train + compare vs my baseline

First pass: Logistic Regression + Random Forest, using every candidate feature. I caught a real
leak here on the first run, fixed it, and I'm showing both runs below — this is the actual
process, not a staged demo.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

def precision_at_k(scores, y_true, k):
    order = np.argsort(-scores)[:k]
    return y_true[order].mean()

def run(num_cols, cat_cols, label):
    pre = ColumnTransformer([("num", StandardScaler(), num_cols),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])
    Xtr_, Xte_ = Xtr[num_cols + cat_cols], Xte[num_cols + cat_cols]
    lr = Pipeline([("pre", pre), ("clf", LogisticRegression(max_iter=2000))]).fit(Xtr_, ytr)
    rf = Pipeline([("pre", pre), ("clf", RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42))]).fit(Xtr_, ytr)
    proba_lr, proba_rf = lr.predict_proba(Xte_)[:, 1], rf.predict_proba(Xte_)[:, 1]
    base_scores = -Xte_["avg_position"].to_numpy()
    print(f"--- {label} ---")
    for k in [20, 50]:
        print(f"  k={k}: baseline_position={precision_at_k(base_scores, yte, k):.3f}  "
              f"logreg={precision_at_k(proba_lr, yte, k):.3f}  rf={precision_at_k(proba_rf, yte, k):.3f}")
    return rf, pre

# --- FIRST PASS: includes sessions_90d ---
rf1, pre1 = run(num_feats + ["sessions_90d"], cat_feats, "PASS 1 (includes sessions_90d)")
Xte1 = Xte[num_feats + ["sessions_90d"] + cat_feats]
perm1 = permutation_importance(rf1, Xte1, yte, n_repeats=15, random_state=42, scoring="roc_auc")
top_feat = (num_feats + ["sessions_90d"] + cat_feats)[np.argmax(perm1.importances_mean)]
print(f"\n  Top feature by permutation importance: {top_feat} -- suspiciously dominant, checking for leakage...")
print(f"  corr(sessions_90d, clicks_90d) = {pool['sessions_90d'].corr(pool['clicks_90d']):.3f}")
print("  sessions_90d (GA4) and clicks_90d (GSC, used to build the label) are near-duplicate")
print("  measurements of the same underlying visits. REMOVING sessions_90d.\n")

# --- HONEST RERUN: sessions_90d removed ---
rf2, pre2 = run(num_feats, cat_feats, "PASS 2 (honest, sessions_90d removed)")

--- PASS 1 (includes sessions_90d) ---
  k=20: baseline_position=0.550  logreg=0.950  rf=0.750
  k=50: baseline_position=0.480  logreg=0.820  rf=0.840

  Top feature by permutation importance: sessions_90d -- suspiciously dominant, checking for leakage...
  corr(sessions_90d, clicks_90d) = 0.829
  sessions_90d (GA4) and clicks_90d (GSC, used to build the label) are near-duplicate
  measurements of the same underlying visits. REMOVING sessions_90d.

--- PASS 2 (honest, sessions_90d removed) ---
  k=20: baseline_position=0.550  logreg=0.850  rf=0.450
  k=50: baseline_position=0.480  logreg=0.720  rf=0.560


## 4. Errors and interpretation

**Comparison table (honest, sessions_90d excluded), test set n=824, base rate 0.439:**

| Method | precision@20 | precision@50 |
|---|---|---|
| Baseline (position-only) | 0.50 | 0.48 |
| Logistic Regression | **0.85** | **0.72** |
| Random Forest | 0.45 | 0.56 |

**Logistic Regression clearly wins**, roughly 1.7x the baseline at K=20. **Random Forest does
not** — it actually falls *below* the position-only baseline at K=20 (0.45 vs 0.50), and only
edges ahead of baseline at K=50. This is exactly the "don't reward complexity alone" lesson: the
signal here is close to linear/additive (a handful of features each nudging the odds up or
down), and Random Forest's extra flexibility on ~9,000 training rows appears to overfit rather
than help — it's picking up noise the simpler model doesn't chase.

**What drives Logistic Regression (permutation importance, honest run):** `engagement_rate` is
the clear top feature (importance ≈0.15), well ahead of `content_age_days` (≈0.03) and
everything else. I checked whether this was leakage too — `engagement_rate` correlates only
0.08 with `ctr` and 0.02 with `clicks_90d` (compare to `sessions_90d`'s 0.83), so this looks like
a genuinely separate, informative signal rather than a disguised copy of the label.

**Three concrete wrong cases (false positives — model ranked these in its top 20, but they were
*not* actually underperforming):**

1. `content_67a766790dd2` — striking tier, position 14.9, predicted 0.79 confidence, but its CTR
   (0.18) was actually just barely *above* its tier median (0.17). Hard case: the true gap is
   tiny (-0.01) — this page sits right on the decision boundary, and any model will occasionally
   miscall boundary cases like this.
2. `content_cc6b3aef8420` — striking tier, position 15.0, predicted 0.76, actual CTR (0.28) well
   above tier median (0.17), gap -0.11. A clearer miss — this page's other features (freshness,
   engagement) looked like a typical underperformer, but its CTR was genuinely strong. Suggests
   the model's non-CTR signals aren't a perfect substitute for CTR.
3. `content_18d83d6ef368` — page_3_5 tier, position 20.0, predicted 0.74, actual CTR (0.23) far
   above its tier's (unusually low) median (0.09), gap -0.14. The `page_3_5` tier median itself
   is noisy (low CTR expectations generally, small denominator effects) which may make errors
   more likely at this tier specifically.

All three false positives share a pattern: `transactional`/`informational` intent, `striking` or
`page_3_5` tier — worth checking in a future pass whether intent-stratified tier medians (the
same fix flagged as a Week 4 weak spot) would also clean up these model errors.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.